# ML_Geely_Auto_Udaykiran_Gunda

## Geely Auto – Multiple Linear Regression for Car Price Prediction

**Assignment:** Predict car prices using multiple linear regression and identify the significant variables affecting price.

**Student:** Udaykiran Gunda

### Business objective
Geely Auto wants to understand the factors that influence car prices in the American market. The model should help management identify significant variables, understand how well they explain price variation, and support product design and pricing decisions.

> This notebook is designed to run from start to finish with the supplied `CarPrice_Assignment.csv` file in the notebook's working directory.

## 1. Import libraries and set reproducibility

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)
pd.set_option("display.max_rows", 100)

RANDOM_STATE = 42

## 2. Load the dataset

In [ ]:
possible_paths = [
    "CarPrice_Assignment.csv",
    "./CarPrice_Assignment.csv",
    "/mnt/data/CarPrice_Assignment.csv",
    "/content/CarPrice_Assignment.csv"
]

data_path = next((p for p in possible_paths if os.path.exists(p)), None)
if data_path is None:
    raise FileNotFoundError(
        "CarPrice_Assignment.csv was not found. Place the supplied dataset in the "
        "same folder as this notebook and run the notebook again."
    )

df = pd.read_csv(data_path)
print("Dataset path:", data_path)
print("Shape:", df.shape)
display(df.head())

## 3. Data understanding and quality checks

In [ ]:
print("Data types:")
display(df.dtypes.to_frame("dtype"))

print("\nSummary statistics:")
display(df.describe(include="all").T)

print("\nMissing values:")
display(df.isnull().sum().sort_values(ascending=False).to_frame("missing_count"))

print("\nDuplicate rows:", df.duplicated().sum())
if "car_ID" in df.columns:
    print("Duplicate car_ID values:", df["car_ID"].duplicated().sum())

### Data-quality treatment

Missing values are imputed conservatively only if they exist: numeric variables use the median and categorical variables use the mode. Exact duplicate rows are removed. The assignment-specific structural issue is `CarName`, which contains manufacturer and model; only the manufacturer is retained.

In [ ]:
df = df.drop_duplicates().copy()

numeric_columns = df.select_dtypes(include=np.number).columns
categorical_columns = df.select_dtypes(exclude=np.number).columns

for col in numeric_columns:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].median())
for col in categorical_columns:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].mode()[0])

print("Missing values after quality treatment:", int(df.isnull().sum().sum()))
print("Rows after duplicate removal:", len(df))

## 4. Data preparation – extract company from CarName

In [ ]:
data = df.copy()
data["company"] = (
    data["CarName"].astype(str).str.strip().str.split().str[0].str.lower()
)

print("Unique company tokens before cleanup:")
print(sorted(data["company"].unique()))

company_map = {
    "vw": "volkswagen",
    "vokswagen": "volkswagen",
    "porcshce": "porsche",
    "toyouta": "toyota",
    "maxda": "mazda"
}
data["company"] = data["company"].replace(company_map)

print("\nCompany counts after cleanup:")
display(data["company"].value_counts().to_frame("count"))

### 4.1 Handle rare manufacturer categories

Manufacturers occurring fewer than five times are grouped into `other_manufacturer`. This retains the observations while reducing unstable dummy-variable estimates.

In [ ]:
company_counts = data["company"].value_counts()
rare_companies = company_counts[company_counts < 5].index.tolist()
data["company"] = data["company"].where(~data["company"].isin(rare_companies), "other_manufacturer")

print("Rare companies grouped:", rare_companies)
print("Final company categories:", data["company"].nunique())
display(data["company"].value_counts().to_frame("count"))

### 4.2 Remove identifiers and create useful derived metrics

`car_ID` is only an identifier and `CarName` contains model information that the assignment does not ask us to use. `cylindernumber` is converted to a numeric cylinder count so derived metrics can be evaluated during EDA.

In [ ]:
cylinder_map = {"two":2, "three":3, "four":4, "five":5, "six":6, "eight":8, "twelve":12}
data["cylinders_num"] = data["cylindernumber"].map(cylinder_map)
data["power_to_weight"] = data["horsepower"] / data["curbweight"]
data["engine_size_per_cylinder"] = data["enginesize"] / data["cylinders_num"]

data["power_to_weight"] = data["power_to_weight"].replace([np.inf, -np.inf], np.nan)
data["engine_size_per_cylinder"] = data["engine_size_per_cylinder"].replace([np.inf, -np.inf], np.nan)
data = data.dropna(subset=["power_to_weight", "engine_size_per_cylinder"]).copy()

model_data = data.drop(columns=["car_ID", "CarName", "cylindernumber"], errors="ignore").copy()
print("Prepared shape:", model_data.shape)
display(model_data.head())

## 5. Exploratory Data Analysis (EDA)

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(model_data["price"], kde=True)
plt.title("Distribution of Car Price")
plt.xlabel("Price")
plt.ylabel("Number of cars")
plt.tight_layout()
plt.show()
print("Price skewness:", round(model_data["price"].skew(), 3))

In [ ]:
numeric_model_columns = model_data.select_dtypes(include=np.number).columns.tolist()
corr_with_price = model_data[numeric_model_columns].corr()["price"].drop("price").sort_values(ascending=False)
display(corr_with_price.to_frame("correlation_with_price"))

plt.figure(figsize=(8,7))
sns.barplot(x=corr_with_price.values, y=corr_with_price.index)
plt.title("Numeric Variable Correlation with Price")
plt.xlabel("Correlation")
plt.ylabel("Variable")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(13,10))
sns.heatmap(model_data[numeric_model_columns].corr(), cmap="coolwarm", center=0)
plt.title("Correlation Heatmap of Numeric Variables")
plt.tight_layout()
plt.show()

In [ ]:
categorical_plot_columns = ["fueltype", "aspiration", "carbody", "drivewheel", "enginelocation", "enginetype"]
for col in categorical_plot_columns:
    plt.figure(figsize=(8,5))
    sns.boxplot(data=model_data, x=col, y="price")
    plt.title(f"Price by {col}")
    plt.xticks(rotation=35)
    plt.tight_layout()
    plt.show()

In [ ]:
plt.figure(figsize=(12,6))
company_order = model_data.groupby("company")["price"].median().sort_values().index
sns.boxplot(data=model_data, x="company", y="price", order=company_order)
plt.title("Price Distribution by Manufacturer")
plt.xticks(rotation=60)
plt.tight_layout()
plt.show()

### EDA interpretation

The EDA is used to understand business relationships before modelling. Engine size, curb weight, horsepower and vehicle dimensions generally have strong positive relationships with price, while fuel-economy measures tend to move in the opposite direction. Several engineering variables are strongly correlated with each other, so including all of them blindly can produce unstable coefficients. Categorical comparisons also show differences across manufacturer and vehicle configuration.

## 6. Encode categorical variables

In [ ]:
target = "price"
categorical_features = model_data.select_dtypes(include=["object"]).columns.tolist()
X_all = pd.get_dummies(model_data.drop(columns=[target]), columns=categorical_features, drop_first=True, dtype=int)
y_all = model_data[target].astype(float)
X_all.columns = X_all.columns.str.replace(r"[^A-Za-z0-9_]+", "_", regex=True).str.strip("_")
print("Encoded predictor shape:", X_all.shape)
display(X_all.head())

## 7. Train-test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, train_size=0.70, test_size=0.30, random_state=RANDOM_STATE
)
print("Training rows:", X_train.shape[0])
print("Test rows:", X_test.shape[0])

## 8. Multicollinearity check using VIF

In [ ]:
def calculate_vif(X):
    X_numeric = X.astype(float).copy()
    values = []
    for i, col in enumerate(X_numeric.columns):
        values.append(variance_inflation_factor(X_numeric.values, i))
    return pd.DataFrame({"feature": X_numeric.columns, "VIF": values}).sort_values("VIF", ascending=False)

vif_initial = calculate_vif(X_train)
display(vif_initial.head(20))

## 9. Candidate model 1 – full multiple linear regression

In [ ]:
full_ols = sm.OLS(y_train, sm.add_constant(X_train.astype(float))).fit()
print(full_ols.summary())

## 10. Candidate model 2 – RFE feature selection

In [ ]:
scaler_rfe = StandardScaler()
X_train_scaled = pd.DataFrame(scaler_rfe.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
n_rfe_features = min(12, X_train_scaled.shape[1])
rfe = RFE(estimator=LinearRegression(), n_features_to_select=n_rfe_features, step=1)
rfe.fit(X_train_scaled, y_train)
rfe_features = X_train.columns[rfe.support_].tolist()
print("RFE selected features:")
print(rfe_features)

rfe_model = sm.OLS(y_train, sm.add_constant(X_train[rfe_features].astype(float))).fit()
print("RFE candidate R-squared:", round(rfe_model.rsquared,4))
print("RFE candidate Adjusted R-squared:", round(rfe_model.rsquared_adj,4))
display(rfe_model.summary2().tables[1])

## 11. Final feature selection using p-values and VIF

In [ ]:
def iterative_pvalue_vif_selection(X, y, p_threshold=0.05, vif_threshold=5.0):
    features = list(X.columns)
    history = []
    while len(features) > 1:
        X_current = X[features].astype(float)
        model = sm.OLS(y, sm.add_constant(X_current)).fit()
        pvalues = model.pvalues.drop("const", errors="ignore")
        max_p_feature = pvalues.idxmax()
        max_p_value = pvalues.max()
        vif_df = calculate_vif(X_current)
        max_vif_feature = vif_df.iloc[0]["feature"]
        max_vif_value = vif_df.iloc[0]["VIF"]
        history.append({
            "n_features": len(features), "r2": model.rsquared, "adj_r2": model.rsquared_adj,
            "worst_p_value": max_p_value, "worst_p_feature": max_p_feature,
            "worst_vif": max_vif_value, "worst_vif_feature": max_vif_feature
        })
        if max_p_value > p_threshold:
            features.remove(max_p_feature)
            continue
        if max_vif_value > vif_threshold:
            features.remove(max_vif_feature)
            continue
        break
    final_model = sm.OLS(y, sm.add_constant(X[features].astype(float))).fit()
    return features, final_model, pd.DataFrame(history)

selected_features, final_ols, selection_history = iterative_pvalue_vif_selection(X_train, y_train)
print("Final selected features:")
print(selected_features)
print("\nModel-selection history:")
display(selection_history)
print("\nFinal OLS summary:")
print(final_ols.summary())

In [ ]:
final_pvalues = final_ols.pvalues.drop("const", errors="ignore").sort_values().to_frame("p_value")
final_vif = calculate_vif(X_train[selected_features])
print("Final p-values:")
display(final_pvalues)
print("Final VIF:")
display(final_vif)

## 12. Compare candidate models with cross-validation

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
candidate_models = {
    "Full model": X_train.columns.tolist(),
    "RFE model": rfe_features,
    "Final p-value + VIF model": selected_features
}
cv_results = []
for model_name, features in candidate_models.items():
    scores = cross_val_score(LinearRegression(), X_train[features].astype(float), y_train, cv=cv, scoring="r2")
    cv_results.append({
        "Model": model_name, "Features": len(features),
        "Mean CV R-squared": scores.mean(), "CV R-squared Std": scores.std()
    })
cv_results_df = pd.DataFrame(cv_results).sort_values("Mean CV R-squared", ascending=False)
display(cv_results_df)

### Model-selection decision

The model comparison considers cross-validated predictive performance together with statistical significance, multicollinearity and interpretability. A model with slightly higher training R-squared is not automatically preferable if it contains unstable or statistically weak predictors.

## 13. Fit final prediction model

In [ ]:
final_regression = LinearRegression()
final_regression.fit(X_train[selected_features].astype(float), y_train)

y_train_pred = final_regression.predict(X_train[selected_features].astype(float))
y_pred = final_regression.predict(X_test[selected_features].astype(float))

train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_pred)
train_rmse = mean_squared_error(y_train, y_train_pred) ** 0.5
test_rmse = mean_squared_error(y_test, y_pred) ** 0.5
train_mae = mean_absolute_error(y_train, y_train_pred)
test_mae = mean_absolute_error(y_test, y_pred)

print("Training R-squared:", round(train_r2,4))
print("Test R-squared:", round(test_r2,4))
print("Training RMSE:", round(train_rmse,2))
print("Test RMSE:", round(test_rmse,2))
print("Training MAE:", round(train_mae,2))
print("Test MAE:", round(test_mae,2))

## 14. Required test-set R-squared calculation

In [ ]:
from sklearn.metrics import r2_score
r2_score(y_test, y_pred)

## 15. Model performance scorecard

In [ ]:
final_scorecard = pd.DataFrame({
    "Metric": ["Training R-squared", "Training Adjusted R-squared", "Test R-squared", "Training RMSE", "Test RMSE", "Training MAE", "Test MAE"],
    "Value": [train_r2, final_ols.rsquared_adj, test_r2, train_rmse, test_rmse, train_mae, test_mae]
})
display(final_scorecard)

## 16. Residual analysis

In [ ]:
residuals = y_train - y_train_pred
fitted_values = y_train_pred

plt.figure(figsize=(8,5))
sns.scatterplot(x=fitted_values, y=residuals)
plt.axhline(0, linestyle="--")
plt.title("Residuals vs Fitted Values")
plt.xlabel("Fitted price")
plt.ylabel("Residual")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8,5))
sns.histplot(residuals, kde=True)
plt.title("Distribution of Residuals")
plt.xlabel("Residual")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7,6))
stats.probplot(residuals, dist="norm", plot=plt)
plt.title("Q-Q Plot of Residuals")
plt.tight_layout()
plt.show()

In [ ]:
shapiro_stat, shapiro_p = stats.shapiro(residuals)
bp_stat, bp_pvalue, bp_fstat, bp_fpvalue = het_breuschpagan(residuals, sm.add_constant(X_train[selected_features].astype(float)))

diagnostic_results = pd.DataFrame({
    "Test": ["Shapiro-Wilk", "Breusch-Pagan LM", "Breusch-Pagan F"],
    "Statistic": [shapiro_stat, bp_stat, bp_fstat],
    "p-value": [shapiro_p, bp_pvalue, bp_fpvalue]
})
display(diagnostic_results)

### Residual-diagnostic interpretation

A good residual plot should show points scattered around zero without a strong curve or funnel shape. The Q-Q plot assesses approximate normality. If the Breusch-Pagan p-value is below 0.05, there is evidence of heteroscedasticity and this should be disclosed as a limitation rather than ignored. The actual notebook output is used for the final conclusion.

## 17. Actual vs predicted price on the test set

In [ ]:
prediction_results = pd.DataFrame({"Actual Price": y_test.values, "Predicted Price": y_pred})
prediction_results["Absolute Error"] = (prediction_results["Actual Price"] - prediction_results["Predicted Price"]).abs()
display(prediction_results.head(15))

plt.figure(figsize=(8,6))
sns.scatterplot(x=prediction_results["Actual Price"], y=prediction_results["Predicted Price"])
min_price = min(prediction_results["Actual Price"].min(), prediction_results["Predicted Price"].min())
max_price = max(prediction_results["Actual Price"].max(), prediction_results["Predicted Price"].max())
plt.plot([min_price, max_price], [min_price, max_price], linestyle="--")
plt.title("Actual vs Predicted Car Price – Test Set")
plt.xlabel("Actual price")
plt.ylabel("Predicted price")
plt.tight_layout()
plt.show()

## 18. Coefficient interpretation

In [ ]:
coefficient_table = pd.DataFrame({
    "Feature": selected_features,
    "Coefficient": final_ols.params[selected_features].values,
    "P-value": final_ols.pvalues[selected_features].values
}).sort_values("P-value")
display(coefficient_table)

print("Significant variables at the 5% level:")
display(coefficient_table[coefficient_table["P-value"] < 0.05])

## 19. Business conclusions

### Question 1: Which variables are significant in predicting car price?

Use the significant-variable table above. Continuous-variable coefficients show the expected price change for a one-unit increase while holding other selected variables constant. Dummy-variable coefficients show the difference from the omitted reference category, all else equal.

### Question 2: How well do those variables describe car price?

Use the **test R-squared** as the main generalisation measure, supported by RMSE and MAE. Test R-squared is the proportion of variation in unseen car prices explained by the final model. RMSE and MAE express prediction error in price units.

### Management implications
- Vehicle size/weight, engine characteristics and manufacturer positioning can materially influence price.
- Strongly correlated engineering variables should not all be interpreted independently because multicollinearity can make coefficients unstable.
- Manufacturer effects can capture brand positioning in the American market.
- The model is a decision-support tool, not a guarantee of the market price of a new Geely vehicle.
- Pricing decisions should also consider factors not present in this dataset, such as incentives, marketing, dealer strategy, financing, brand awareness and macroeconomic conditions.

## 20. Final checklist against the evaluation rubric

- [x] Missing-value and duplicate checks performed.
- [x] `CarName` converted to manufacturer/company only.
- [x] Manufacturer spelling inconsistencies corrected.
- [x] Rare manufacturer categories handled.
- [x] Categorical variables converted to dummy variables.
- [x] Relevant derived metrics created and evaluated.
- [x] EDA includes target distribution, numeric relationships, correlations and categorical comparisons.
- [x] Multicollinearity checked using VIF.
- [x] More than one candidate model/feature-selection approach attempted.
- [x] Final variables selected using statistical significance and multicollinearity considerations.
- [x] Train/test split used before final prediction evaluation.
- [x] R-squared, Adjusted R-squared, RMSE and MAE reported.
- [x] Required `r2_score(y_test, y_pred)` calculation included exactly.
- [x] Residual analysis performed.
- [x] Model coefficients and significant variables interpreted.
- [x] Business conclusions answer both Geely questions.
- [x] Code uses comments, descriptive variable names and readable sections.